# **인프라 시설 총점 (Total Score) 계산**

## 목적

여러 인프라 시설별 거리 데이터를 활용하여, 각 후보지 주변에서 **가장 가까운 5개 시설**을 기준으로 거리 기반 총점을 계산합니다.

- 각 시설별 **적정 거리**를 기준으로 시그모이드 함수를 적용해 거리별 영향력을 반영합니다.
- 가까운 시설일수록 높은 점수를 부여하되, 5개 시설의 점수가 단순 합산되어 과도하게 커지지 않도록 감쇠 가중치를 적용합니다.
- 대용량 데이터를 **청크 단위로 나누어 처리**하여 메모리 사용을 최적화합니다.

# 1.라이브러리 및 환경 설정

In [ ]:
# 필수 라이브러리 설치
!pip install koreanize_matplotlib factor_analyzer

# 기본 라이브러리 임포트
import pandas as pd
import numpy as np
import glob
import os
import platform
import matplotlib.pyplot as plt
import koreanize_matplotlib
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
from sklearn.metrics import silhouette_score
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo

# 한글 폰트 설정
from matplotlib import font_manager, rc

if platform.system() == 'Windows':
    font_name = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    font_name = 'AppleGothic'
else:  # Linux
    font_name = 'NanumGothic'

rc('font', family=font_name)
plt.rcParams['axes.unicode_minus'] = False

print("설정된 폰트:", plt.rcParams['font.family'])


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 40.2 MB/s eta 0:00:00
  Created wheel for factor_analyzer: filename=factor_analyzer-0.5.1-py2.py3-none-any.whl size=42655 sha256=9be57a9db596eaf17faa1c79c65c0e9275bc919a9e80f0f279710db0652b7492
  Stored in directory: /root/.cache/pip/wheels/fa/f7/53/a55a8a56668a6fe0199e0e02b6e0ae3007ec35cdf6e4c25df7
Successfully built factor_analyzer
설정된 폰트: ['NanumGothic']


# 2.구글 드라이브 연동 및 작업 디렉토리 설정

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
os.chdir("/content/drive/MyDrive/비어플/dataset")
print("현재 작업 디렉토리:", os.getcwd())

Mounted at /content/drive
현재 작업 디렉토리: /content/drive/MyDrive/비어플/dataset


# 3.변수 및 상수 선언

## 3-1.인프라 시설명 매핑 딕셔너리

In [ ]:
facility_name_map = {
    'bank': "은행",
    'bus_station': "버스정류장",
    'car_center': "차량정비소",
    'conve': "편의점",
    'express_station': "고속버스터미널",
    'farmtech_center': "농업기술센터",
    'farm_market': "농산물직거래판매장",
    'farm': "농지매물",
    'lesson': "학원",
    'post_office': "우체국",
    'police': "경찰서",
    'rental': "농기계대여소",
    'school': "초중학교",
    'store': "시장",
    'town_center': "행정복지센터",
    'train_station': "기차역",
    'hospital': "병원",
}

## 3-2.시설별 적정 거리 (단위: km 등, 분석 목적에 맞게)

In [ ]:
# 시설별 적정 거리 기준 (단위: km)
dist_map = {
    "초중학교":       1,
    "학원":           1,
    "병원":           5,
    "은행":           5,
    "편의점":         1,
    "시장":           3,
    "버스정류장":     1,
    "고속버스터미널": 7.5,
    "기차역":         7.5,
    "경찰서":         5,
    "행정복지센터":   5,
    "우체국":         3,
    "농지매물":       2,
    "차량정비소":     5,
    "농업기술센터":   10,
    "농산물직거래판매장": 10,
    "농기계대여소":   5,
}

# 4.데이터 파일 목록 가져오기

In [ ]:
csv_file_paths = glob.glob('좌표거리데이터/result_house_*.csv')
print(f"총 {len(csv_file_paths)}개 파일을 찾았습니다.")

총 17개 파일을 찾았습니다.


# 5.거리 기반 총점 계산 함수 정의

### 1. Sigmoid 기반 거리 점수 계산

각 거리 `d`에 대해 다음과 같은 점수를 부여합니다:

```
score(d) = 1 / (1 + exp(b * (d - m)))
```

- `d`: 해당 후보지에서 특정 시설까지의 거리  
- `m`: 해당 시설의 **적정 거리 기준**  
- 결과적으로 거리가 가까울수록 높은 점수

---

### 2. Top-5 감쇠 누적 점수 적용

각 후보지에서 점수가 높은 상위 5개 시설에 대해 다음과 같이 감쇠 계수를 적용한 총점을 계산합니다:

```
Total Score = score₀ × decay⁰ + score₁ × decay¹ + score₂ × decay² + score₃ × decay³ + score₄ × decay⁴
```

- `scoreᵢ`: 거리 점수 중 `i`번째로 높은 값 (최대 5개)
- `decay`: 감쇠 계수 (예: `0.9`)
- 가까운 시설일수록 높은 가중치가 부여되고 순위가 낮을수록 영향이 줄어듦

---

In [ ]:
def process_file_row_chunks(file_path, chunk_row_size=500):
    """
    거리 기반 감쇠 점수를 계산하여 상위 5개 항목만 반영한 총점(total_score)을 반환합니다.

    Parameters:
        file_path (str): 분석할 CSV 파일 경로
        chunk_row_size (int): 청크 크기 (기본값: 500)

    Returns:
        pd.DataFrame: 'address'와 '[시설명]_total_score' 컬럼을 포함한 결과
    """
    facility_key = file_path.split("result_house_")[1].replace(".csv", "")
    facility_name = facility_name_map.get(facility_key)
    dist = dist_map.get(facility_name)

    m = dist
    b = 2 / m
    decay = 0.9

    preview = pd.read_csv(file_path, encoding='utf-8-sig', nrows=5)
    numeric_cols = preview.select_dtypes(include=[np.number]).columns.tolist()
    dtype_map = {col: np.float32 for col in numeric_cols}

    partial_results = []

    for chunk in pd.read_csv(file_path, encoding='utf-8-sig', dtype=dtype_map, chunksize=chunk_row_size):
        chunk = chunk.rename(columns={chunk.columns[0]: 'address'})
        dist_cols = chunk.select_dtypes(include=[np.number]).columns

        distances = chunk[dist_cols]
        exp_input = b * (distances - m)
        safe_exp_input = np.clip(exp_input, -50, 50)
        weights = 1 / (1 + np.exp(safe_exp_input))

        def apply_decay_top5(row):
            sorted_weights = sorted(row.dropna(), reverse=True)[:5]
            return sum((decay ** i) * w for i, w in enumerate(sorted_weights))

        total_score_values = weights.apply(apply_decay_top5, axis=1)
        total_col = facility_name + '_total_score'
        chunk[total_col] = total_score_values

        partial_results.append(chunk[['address', total_col]])

    return pd.concat(partial_results, ignore_index=True)

# 6.여러 파일에 대해 근접 점수 계산 및 병합

In [ ]:
total_score_df = None

for path in csv_file_paths:
    temp_df = process_file_row_chunks(path)

    if total_score_df is None:
        total_score_df = temp_df.copy()
    else:
        score_col = temp_df.columns[-1]
        total_score_df[score_col] = temp_df[score_col]

    print(f"{path} 처리 완료!")

# 최종 결과 저장
total_score_df.to_csv('facility_total_scores.csv', index=False, encoding='utf-8-sig')

좌표거리데이터/result_house_post_office.csv 처리 완료!
좌표거리데이터/result_house_police.csv 처리 완료!
좌표거리데이터/result_house_farmtech_center.csv 처리 완료!
좌표거리데이터/result_house_farm_market.csv 처리 완료!
좌표거리데이터/result_house_farm.csv 처리 완료!
좌표거리데이터/result_house_conve.csv 처리 완료!
좌표거리데이터/result_house_lesson.csv 처리 완료!
좌표거리데이터/result_house_bank.csv 처리 완료!
좌표거리데이터/result_house_train_station.csv 처리 완료!
좌표거리데이터/result_house_car_center.csv 처리 완료!
좌표거리데이터/result_house_town_center.csv 처리 완료!
좌표거리데이터/result_house_store.csv 처리 완료!
좌표거리데이터/result_house_rental.csv 처리 완료!
좌표거리데이터/result_house_school.csv 처리 완료!
좌표거리데이터/result_house_bus_station.csv 처리 완료!
좌표거리데이터/result_house_hospital.csv 처리 완료!
좌표거리데이터/result_house_express_station.csv 처리 완료!
